# NOTEBOOK : 04_player_evaluation_save.ipynb
# INPUT    : ../artifacts/models/player_candidates.joblib, ../../data/processed/player_features.csv
# OUTPUT   : ../artifacts/models/player_model.joblib, ../artifacts/metrics.json

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path("../..")
PROCESSED_DIR = ROOT / "data" / "processed"
ARTIFACTS_DIR = Path("../artifacts")
MODELS_DIR = ARTIFACTS_DIR / "models"
METRICS_PATH = ARTIFACTS_DIR / "metrics.json"

bundle = joblib.load(MODELS_DIR / "player_candidates.joblib")
models = bundle["models"]
scaler = bundle["scaler"]
feature_cols = bundle["feature_cols"]

df = pd.read_csv(PROCESSED_DIR / "player_features.csv", low_memory=False)
test_df = df[df["season"] == "2025-26"].copy().reset_index(drop=True)

X_test_sc = scaler.transform(test_df[feature_cols].values)
y_test = test_df["target_points"].values

In [2]:
def compute_star_hit_rates(eval_df: pd.DataFrame, pred_col: str) -> tuple[float, float]:
    """Top-1 & Top-3 match-star hit rate grouped by (season, round, team)."""
    group_cols = ["season", "round", "team"]
    top1_hits, top3_hits, total_groups = 0, 0, 0

    for _, grp in eval_df.groupby(group_cols):
        if len(grp) < 3:
            continue
        actual_top_idx = grp["target_points"].idxmax()
        pred_top3_idx = grp.nlargest(3, pred_col).index.tolist()
        if actual_top_idx == pred_top3_idx[0]:
            top1_hits += 1
        if actual_top_idx in pred_top3_idx:
            top3_hits += 1
        total_groups += 1

    if total_groups == 0:
        return 0.0, 0.0
    return round(top1_hits / total_groups, 4), round(top3_hits / total_groups, 4)

In [3]:
def evaluate_predictions(name: str, y_pred: np.ndarray) -> dict:
    mse = mean_squared_error(y_test, y_pred)
    eval_df = test_df.copy()
    eval_df["pred"] = y_pred
    top1, top3 = compute_star_hit_rates(eval_df, "pred")
    return {
        "model": name,
        "mae": round(float(mean_absolute_error(y_test, y_pred)), 4),
        "rmse": round(float(np.sqrt(mse)), 4),
        "mse": round(float(mse), 4),
        "r2": round(float(r2_score(y_test, y_pred)), 4),
        "top1_star_hit_rate": top1,
        "top3_star_hit_rate": top3,
    }

In [4]:
results = [evaluate_predictions("rolling_avg_baseline", test_df["total_points_roll5"].values)]
for name, model in models.items():
    results.append(evaluate_predictions(name, model.predict(X_test_sc)))

results_df = pd.DataFrame(results).sort_values("rmse")
print("Test Set Evaluation (2025-26):")
print(results_df.to_string(index=False))

# Select winning ML model (lowest RMSE)
winner_row = results_df[results_df["model"] != "rolling_avg_baseline"].iloc[0].to_dict()
winner_name = winner_row["model"]

joblib.dump(
    {
        "model_name": winner_name,
        "model": models[winner_name],
        "scaler": scaler,
        "feature_cols": feature_cols,
        "metrics": winner_row,
    },
    MODELS_DIR / "player_model.joblib",
)
print(f"\nSaved winning model ({winner_name}) -> {MODELS_DIR / 'player_model.joblib'}")

Test Set Evaluation (2025-26):
               model    mae   rmse    mse     r2  top1_star_hit_rate  top3_star_hit_rate
             xgboost 0.9714 1.9771 3.9088 0.2940              0.1213              0.2787
       random_forest 0.9647 1.9780 3.9123 0.2934              0.0987              0.2933
   linear_regression 0.9990 1.9897 3.9588 0.2850              0.1187              0.2973
               ridge 0.9990 1.9897 3.9588 0.2850              0.1187              0.2973
               lasso 0.9987 1.9906 3.9626 0.2843              0.1173              0.3000
rolling_avg_baseline 1.0520 2.1071 4.4400 0.1981              0.0960              0.2733

Saved winning model (xgboost) -> ..\artifacts\models\player_model.joblib


In [5]:
# Append regression metrics to metrics.json
metrics_data = {}
if METRICS_PATH.exists():
    with open(METRICS_PATH, "r", encoding="utf-8") as f:
        metrics_data = json.load(f)

metrics_data["regression"] = {
    "evaluated_at_utc": datetime.now(timezone.utc).isoformat(),
    "best_model": winner_name,
    "mae": winner_row["mae"],
    "rmse": winner_row["rmse"],
    "mse": winner_row["mse"],
    "r2": winner_row["r2"],
    "top1_star_hit_rate": winner_row["top1_star_hit_rate"],
    "top3_star_hit_rate": winner_row["top3_star_hit_rate"],
    "model_comparison": results_df.to_dict(orient="records"),
}

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics_data, f, indent=2)

print(f"Updated {METRICS_PATH} with regression metrics.")

Updated ..\artifacts\metrics.json with regression metrics.
